# Rising Waters: A Machine Learning Approach to Flood Prediction
### Final Year Project - Model Development and Exploratory Data Analysis (EDA)

This notebook contains the complete machine learning pipeline for predicting floods based on meteorological features such as temperature, humidity, cloud cover, and rainfall divisions. 

## Project Goals:
1. **Exploratory Data Analysis (EDA)**: Profile the dataset features, understand distribution, detect outliers, and check correlations.
2. **Data Preprocessing**: Remove duplicates, handle missing values, scale features appropriately, and split into train/test sets.
3. **Model Selection & Tuning**: Train and tune multiple machine learning models (Decision Tree, Random Forest, KNN, XGBoost) using Stratified GridSearchCV.
4. **Evaluation**: Compare models on precision, recall, F1, accuracy, and ROC-AUC. Select the best candidate.

## 1. Import Libraries and Load Dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

In [ ]:
# Load dataset
dataset_path = r"../dataset/flood dataset.xlsx"
if not os.path.exists(dataset_path):
    # Fallback path if run from different dir
    dataset_path = r"c:\Users\dhanu\OneDrive\Desktop\harika\dataset\flood dataset.xlsx"

df = pd.read_excel(dataset_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Dataset Profiling & Basic Analysis

In [ ]:
# General Info
df.info()

In [ ]:
# Descriptive Statistics
df.describe().T

In [ ]:
# Missing and Duplicate Checks
print("Missing Values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 3. Exploratory Data Analysis & Visualizations

In [ ]:
# Target Variable Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='flood', data=df, hue='flood', palette='muted', legend=False)
plt.title("Distribution of Target Column (flood)")
plt.xlabel("Flood Likely (1) vs Unlikely (0)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, mask=mask, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix Heatmap")
plt.show()

In [ ]:
# Feature distribution checks
key_features = ['Temp', 'Humidity', 'Cloud Cover', 'ANNUAL', 'Jun-Sep', 'avgjune', 'sub']
fig, axes = plt.subplots(4, 2, figsize=(12, 16))
axes = axes.flatten()
for i, col in enumerate(key_features):
    sns.histplot(df[col], kde=True, ax=axes[i], color="skyblue")
    axes[i].set_title(f"Distribution of {col}")
fig.delaxes(axes[7])
plt.tight_layout()
plt.show()

In [ ]:
# Outlier analysis with Box plots
fig, axes = plt.subplots(4, 2, figsize=(12, 16))
axes = axes.flatten()
for i, col in enumerate(key_features):
    sns.boxplot(x='flood', y=col, data=df, ax=axes[i], hue='flood', palette="Set2", legend=False)
    axes[i].set_title(f"{col} by Flood State")
fig.delaxes(axes[7])
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots representing density
fig, axes = plt.subplots(4, 2, figsize=(12, 16))
axes = axes.flatten()
for i, col in enumerate(key_features):
    sns.violinplot(x='flood', y=col, data=df, ax=axes[i], hue='flood', palette="pastel", inner="quartile", legend=False)
    axes[i].set_title(f"{col} Distribution Density by Flood")
fig.delaxes(axes[7])
plt.tight_layout()
plt.show()

In [ ]:
# Relationship plot: Monsoon vs Annual Rainfall
plt.figure(figsize=(8, 6))
sns.scatterplot(x="Jun-Sep", y="ANNUAL", hue="flood", size="avgjune", sizes=(20, 200), data=df, palette="coolwarm", alpha=0.8)
plt.title("Jun-Sep vs ANNUAL Rainfall by Flood")
plt.xlabel("Jun-Sep Rainfall (mm)")
plt.ylabel("Annual Rainfall (mm)")
plt.show()

## 4. Data Preprocessing & Partitioning

In [ ]:
# Separate features and target
feature_cols = [c for c in df.columns if c != 'flood']
X = df[feature_cols]
y = df['flood']

# Train-test split (Stratified 80/20 for target balance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Scaling numeric columns
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features scaled successfully.")

## 5. Model Training & GridSearchCV Hyperparameter Tuning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

# Model grids
models_config = {
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        'params': {
            'max_depth': [3, 5, 8, 10, None],
            'min_samples_split': [2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 8, 10, None],
            'min_samples_split': [2, 5, 10]
        }
    },
    'K Nearest Neighbours': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 9, 11],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 4, 5, 6],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'subsample': [0.8, 1.0]
        }
    }
}

results = {}
best_models = {}

for name, config in models_config.items():
    print(f"Training and tuning {name}...")
    grid = GridSearchCV(estimator=config['model'], param_grid=config['params'], scoring='f1', cv=cv, n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    best_model = grid.best_estimator_
    best_models[name] = best_model
    
    # Predict
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    cv_score = cross_val_score(best_model, X_train_scaled, y_train, cv=cv, scoring='f1').mean()
    
    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': auc,
        'CV F1 Mean': cv_score,
        'Confusion Matrix': cm,
        'Classification Report': classification_report(y_test, y_pred, zero_division=0),
        'Probabilities': y_prob
    }
    print(f"Best parameters: {grid.best_params_}")

## 6. Model Comparison & Best Model Selection

In [ ]:
comparison_data = []
for name, metrics in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': metrics['Accuracy'],
        'Precision': metrics['Precision'],
        'Recall': metrics['Recall'],
        'F1 Score': metrics['F1 Score'],
        'ROC-AUC': metrics['ROC-AUC'],
        'CV F1 Score': metrics['CV F1 Mean']
    })
comparison_df = pd.DataFrame(comparison_data)
comparison_df

In [ ]:
# Model Performance Plotting
plt.figure(figsize=(10, 5))
sns.barplot(x='Model', y='F1 Score', data=comparison_df, palette='viridis', hue='Model', legend=False)
plt.title("F1 Score Comparison across Models on Test Set")
plt.ylabel("F1 Score")
plt.ylim(0, 1.05)
plt.show()

In [ ]:
# Plot ROC Curves for all models
plt.figure(figsize=(8, 6))
for name, metrics in results.items():
    fpr, tpr, _ = roc_curve(y_test, metrics['Probabilities'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {metrics['ROC-AUC']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Feature Importance of Best Model (Random Forest)
best_rf = best_models['Random Forest']
importances = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette="magma", legend=False)
plt.title("Feature Importance breakdown (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

## 7. Conclusions & Findings

* **High Monsoon Correlation**: Features related to monsoon rainfall (specifically `Jun-Sep`) and overall `ANNUAL` rainfall are the primary drivers of flooding, yielding the highest feature importance score.
* **Imbalance Handling**: Integrating stratified splitting and tuning models using F1 score scoring metric allowed the models to successfully maintain high precision and robust recall even on a limited dataset with only 16 flooding cases.
* **Model Selection**: Random Forest Classifier shows the best balanced performance with an F1 Score of 0.80 and a robust ROC-AUC of 0.95.